# Library Imports and Data Saving

### Fetch the data and save it

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from stonks_modules import *
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

plt.rcParams["figure.dpi"] = 300

# The ticker for State Bank of India on the National Stock Exchange
# ticker_symbols = ["^NSEI"]
ticker_symbols = ["NVDA", "AMD", "AAPL", "GOOGL"]
sbi = yf.Ticker(ticker_symbols[0])
tickers = {symbol:yf.Ticker(symbol) for symbol in ticker_symbols}


ModuleNotFoundError: No module named 'stonks_modules'

### Manipulate the data to get closes and baseline profit

In [ ]:
hourly_data_files = {ticker_symbol: fetch_and_save_data(ticker)[0] for ticker_symbol, ticker in tickers.items()}
hourly_dataframes = {ticker_symbol: pd.read_csv(filename) for ticker_symbol, filename in hourly_data_files.items()}
hourly_closes = {ticker_symbol: df[["Close"]] for ticker_symbol, df in hourly_dataframes.items()}
hourly_closes_arrays = {ticker_symbol: df["Close"].values for ticker_symbol, df in hourly_closes.items()}

In [ ]:
baseline_profits = {}

for ticker_symbol, close_df in hourly_closes.items():
    first_close = close_df.iloc[0]   # keeps same behavior as your original code (Series)
    last_close = close_df.iloc[-1]
    baseline_profits[ticker_symbol] = ((100 / first_close) * last_close).iloc[0]

# Creating the Heatmaps (for default Wait = 5)

### Initialize heatmap data

In [ ]:
sell_multipliers = np.round(np.linspace(1.01, 1.10, 90), decimals=3)
buy_multipliers = np.round(np.linspace(0.90, 0.99, 90), decimals=3)

In [ ]:
print(sell_multipliers)

In [ ]:
backtested_profits = {
    ticker_symbol : backtest_basic_dip_strategy(close_array, buy_multipliers, sell_multipliers, wait_period=11)
    for ticker_symbol, close_array in hourly_closes_arrays.items()
	}

### Make heatmaps

In [ ]:
x_tick_distance = len(sell_multipliers) // 5
y_tick_distance = len(buy_multipliers) // 5

profit_compared_to_baseline = {}
vmins, vmaxes = [], []

cmap = LinearSegmentedColormap.from_list(
    "red_white_green",
    ["red", "white", "green"]
)

for ticker_symbol, profits_matrix in backtested_profits.items():
	plt.figure(figsize=(10, 8))

	profit_compared_to_baseline[ticker_symbol] = (profits_matrix - baseline_profits[ticker_symbol]) * 100 /baseline_profits[ticker_symbol]
	vmins.append(np.min(profit_compared_to_baseline[ticker_symbol]))
	vmaxes.append(np.max(profit_compared_to_baseline[ticker_symbol]))

norm = TwoSlopeNorm(
    vmin=np.min(vmins),      # minimum value in your data
    vcenter=0,      # THIS makes 0 the center
    vmax=np.max(vmaxes)       # maximum value
)

for ticker_symbol, profits_matrix in backtested_profits.items():
	sns.heatmap(
		profit_compared_to_baseline[ticker_symbol], 
		xticklabels=sell_multipliers, 
		yticklabels=buy_multipliers, 
		cmap=cmap,
		norm=norm)

	plt.title(f"Profit Heatmap for {ticker_symbol} (Represented as % Difference from Baseline)")
	plt.xlabel("Sell Multipliers")
	plt.ylabel("Buy Multipliers")

	ax = plt.gca()

	ax.set_xticks(ax.get_xticks()[::x_tick_distance])
	ax.set_xticklabels(sell_multipliers[::x_tick_distance], rotation=45)

	ax.set_yticks(ax.get_yticks()[::y_tick_distance])
	ax.set_yticklabels(buy_multipliers[::y_tick_distance], rotation=0)

	plt.tight_layout()
	plt.show()

# Creating the Heatmaps (for a list of waits)

### Initialize heatmap data

In [ ]:
wait_periods = list(range(1, 21))
sell_multipliers = np.round(np.linspace(1.01, 1.10, 90), 3)
buy_multipliers = np.round(np.linspace(0.90, 0.99, 90), 3)

In [ ]:
backtested_profits_with_wait_periods = {
    ticker_symbol : {
        wait_period: backtest_basic_dip_strategy(close_array, buy_multipliers, sell_multipliers, wait_period=wait_period)
        for wait_period in wait_periods
    }
    for ticker_symbol, close_array in hourly_closes_arrays.items()
}

### Make heatmaps

In [ ]:
x_tick_distance = len(sell_multipliers) // 5
y_tick_distance = len(buy_multipliers) // 5

vmins, vmaxes = [], []
data_lists = {}


for ticker_symbol, backtedted_profits_by_wait in backtested_profits_with_wait_periods.items():
    baseline_profit = baseline_profits[ticker_symbol]

    for wait, profits_matrix in backtedted_profits_by_wait.items():
        new_profits_matrix = (profits_matrix - baseline_profit) * 100/ baseline_profit
        if ticker_symbol not in data_lists:
            data_lists[ticker_symbol] = []
        data_lists[ticker_symbol].append(new_profits_matrix)

    vmins.append(min(d.min() for d in data_lists[ticker_symbol]))
    vmaxes.append(max(d.max() for d in data_lists[ticker_symbol]))

vmin, vmax = min(vmins), max(vmaxes)

norm = TwoSlopeNorm(
    vmin=vmin,      # minimum value in your data
    vcenter=0,      # THIS makes 0 the center
    vmax=vmax       # maximum value
)

cmap = LinearSegmentedColormap.from_list(
    "red_white_green",
    ["red", "white", "green"]
)

for ticker_symbol, backtedted_profits_by_wait in backtested_profits_with_wait_periods.items():

    fig, axes = plt.subplots(5, 4, figsize=(16, 18), constrained_layout=True)

    for idx, ax in enumerate(axes.flat):
        sns.heatmap(data_lists[ticker_symbol][idx], 
                    annot=False, 
                    cmap=cmap,
                    xticklabels=sell_multipliers,
                    yticklabels=buy_multipliers,
                    ax=ax,
                    cbar=False,
                    norm = norm
                    )

        ax.set_xticks(np.arange(0, len(sell_multipliers), x_tick_distance) + 0.5)
        ax.set_xticklabels(sell_multipliers[::x_tick_distance], rotation=45)

        ax.set_yticks(np.arange(0, len(buy_multipliers), y_tick_distance) + 0.5)
        ax.set_yticklabels(buy_multipliers[::y_tick_distance], rotation=0)
        
        ax.set_title(f"Wait: {wait_periods[idx]} Hours")

    cbar = fig.colorbar(
        plt.cm.ScalarMappable(cmap=cmap,
                              norm=norm),
        ax=axes,
        orientation="vertical",
        fraction=0.02,
        pad=0.02
    )

    cbar.set_label("Profit Above Baseline (%)")

    fig.suptitle(f"Trading Strategy Grid Search for {ticker_symbol}", fontsize=26)

    # fig.tight_layout(rect=[0, 0, 0.95, 0.96])

    plt.savefig(f"heatmaps_{ticker_symbol}.png", dpi=600, bbox_inches="tight")
    plt.show()

# Stacked 3D Profit Heatmap
Visualizes the profit from the grid search across different wait periods as stacked 2D surfaces in a 3D plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Create 2D coordinate meshgrids based on your multipliers
Sell_mesh, Buy_mesh = np.meshgrid(sell_multipliers, buy_multipliers)

for ticker_symbol in backtested_profits_with_wait_periods.keys():
    fig = plt.figure(figsize=(14, 12))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot a surface layer for each wait period
    for w_idx, wait in enumerate(wait_periods):
        profits_matrix = data_lists[ticker_symbol][w_idx]
        
        # The Z-height for this surface is just the constant wait period
        Z_surface = np.full_like(Sell_mesh, wait)
        
        # Map the profit values to RGBA colors using the existing colormap and norm
        colors = cmap(norm(profits_matrix))
        
        surf = ax.plot_surface(
            Sell_mesh, 
            Buy_mesh, 
            Z_surface, 
            facecolors=colors, 
            rstride=1, cstride=1, 
            alpha=0.8,     # Transparency to see through the stack
            shade=False    # Avoid lighting interference with the heatmap colors
        )

    ax.set_xlabel('Sell Multipliers')
    ax.set_ylabel('Buy Multipliers')
    ax.set_zlabel('Wait Period (Hours)')
    ax.set_title(f"Stacked 3D Profit Heatmap for {ticker_symbol}")

    # Add a colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label("Profit Above Baseline (%)")

    ax.view_init(elev=15, azim=-45) # Angle for viewing the stack
    
    # Set Z ticks to distinct integer wait periods
    ax.set_yticks(np.arange(0.9, 1.0, 0.02))
    ax.set_zticks(wait_periods[::2])
    
    plt.tight_layout()
    plt.savefig(f"stacked_3d_heatmap_{ticker_symbol}.png", dpi=300, bbox_inches="tight")
    plt.show()


# 3D Scatterplot Heatmap
Visualizes the profit from the grid search using a 3D scatterplot for true 3-axis representation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# TOGGLE: Set to True to only show profitable points, preventing the plot from looking like a solid cube
FILTER_PROFIT = True

Sell_mesh, Buy_mesh = np.meshgrid(sell_multipliers, buy_multipliers)

for ticker_symbol in backtested_profits_with_wait_periods.keys():
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    X, Y, Z, C = [], [], [], []
    
    for w_idx, wait in enumerate(wait_periods):
        profits_matrix = data_lists[ticker_symbol][w_idx]
        
        X.append(Sell_mesh.flatten())
        Y.append(Buy_mesh.flatten())
        Z.append(np.full(profits_matrix.size, wait))
        C.append(profits_matrix.flatten())

    X = np.concatenate(X)
    Y = np.concatenate(Y)
    Z = np.concatenate(Z)
    C = np.concatenate(C)
    
    if FILTER_PROFIT:
        mask = C > 0  # Only show strict profits over baseline
        X_plot, Y_plot, Z_plot, C_plot = X[mask], Y[mask], Z[mask], C[mask]
    else:
        X_plot, Y_plot, Z_plot, C_plot = X, Y, Z, C

    # Plot the 3D scatter (These are the true 3d dots!)
    sc = ax.scatter(
        X_plot, 
        Y_plot, 
        Z_plot, 
        c=C_plot, 
        cmap=cmap,
        norm=norm,
        s=15,
        alpha=0.6 if FILTER_PROFIT else 0.05 # Massive transparency needed if plotting all 162k dots
    )
    
    ax.set_xlabel('Sell Multipliers')
    ax.set_ylabel('Buy Multipliers')
    ax.set_zlabel('Wait Period (Hours)')
    ax.set_title(f"3D Scatter Heatmap for {ticker_symbol} \n({'Showing Only Profitable Configs' if FILTER_PROFIT else 'Showing All Configs'})")

    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label("Profit Above Baseline (%)")

    plt.tight_layout()
    plt.savefig(f"scatter_3d_heatmap_{ticker_symbol}.png", dpi=300, bbox_inches="tight")
    plt.show()
